# Trackify Behavior Detection - Model Training

**Before running:** Go to `Runtime` → `Change runtime type` → Select **T4 GPU**

This notebook trains a custom YOLOv8s model on your merged classroom behavior dataset.

**Steps:**
1. Upload `merged_dataset.zip` (your merged dataset)
2. Run all cells
3. Download the trained `trackify_behavior.pt` at the end

## Step 1: Check GPU & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Install ultralytics (YOLO)
!pip install -q ultralytics

## Step 2: Upload Dataset

Upload the `merged_dataset.zip` file when prompted below.

In [ ]:
from google.colab import files
import os

print("Upload merged_dataset.zip ...")
uploaded = files.upload()

# Extract
!unzip -q -o merged_dataset.zip -d /content/

# Verify structure
print("\n=== Dataset Structure ===")
!ls /content/merged_dataset/
print("\nTrain images:", len(os.listdir("/content/merged_dataset/train/images")))
print("Val images:", len(os.listdir("/content/merged_dataset/val/images")))
print("Test images:", len(os.listdir("/content/merged_dataset/test/images")))
print("\n=== data.yaml ===")
!cat /content/merged_dataset/data.yaml

In [ ]:
# Fix data.yaml paths for Colab
data_yaml = """
path: /content/merged_dataset
train: train/images
val: val/images
test: test/images

nc: 8
names:
  0: person
  1: phone_use
  2: sleeping
  3: talking
  4: cheating
  5: fighting
  6: eating
  7: drinking
"""

with open("/content/merged_dataset/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml updated for Colab paths")

## Step 3: Base Training

Using **YOLOv8s** (Small) on GPU — should take **20-40 minutes** on T4.

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s pretrained on COCO
model = YOLO("yolov8s.pt")

# Train
results = model.train(
    data="/content/merged_dataset/data.yaml",
    epochs=100,
    batch=16,
    imgsz=640,
    patience=20,           # Early stopping
    project="/content/training",
    name="base_v1",
    optimizer="AdamW",
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,            # Brightness variation for classroom lighting
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
)

print("\n" + "="*50)
print("  BASE TRAINING COMPLETE")
print("="*50)

## Step 4: Evaluate Results

In [ ]:
import os
from IPython.display import Image, display

# Show training results
results_dir = "/content/training/base_v1"

# Display training curves
if os.path.exists(f"{results_dir}/results.png"):
    display(Image(filename=f"{results_dir}/results.png", width=900))

# Display confusion matrix
if os.path.exists(f"{results_dir}/confusion_matrix.png"):
    print("\nConfusion Matrix:")
    display(Image(filename=f"{results_dir}/confusion_matrix.png", width=700))

# Display sample predictions
if os.path.exists(f"{results_dir}/val_batch0_pred.jpg"):
    print("\nSample Predictions:")
    display(Image(filename=f"{results_dir}/val_batch0_pred.jpg", width=900))

In [ ]:
# Run validation to see per-class metrics
best_model = YOLO("/content/training/base_v1/weights/best.pt")
val_results = best_model.val(
    data="/content/merged_dataset/data.yaml",
    imgsz=640,
    batch=16,
)

print("\n" + "="*50)
print(f"  mAP@50:    {val_results.box.map50:.3f}")
print(f"  mAP@50-95: {val_results.box.map:.3f}")
print("="*50)

# Per-class AP
names = {0: 'person', 1: 'phone_use', 2: 'sleeping', 3: 'talking',
         4: 'cheating', 5: 'fighting', 6: 'eating', 7: 'drinking'}
print("\nPer-class AP@50:")
for i, ap in enumerate(val_results.box.ap50):
    print(f"  {names.get(i, f'class_{i}'):12s}: {ap:.3f}")

## Step 5: Fine-tune on Classroom Data (Optional)

This step fine-tunes the model specifically on classroom datasets (D3 + D4).
**Skip this if base training results are already good (mAP@50 > 0.65).**

In [ ]:
# OPTIONAL: Fine-tune on classroom data
# Only run this if you want to squeeze more accuracy

ft_model = YOLO("/content/training/base_v1/weights/best.pt")

ft_results = ft_model.train(
    data="/content/merged_dataset/data.yaml",  # same dataset
    epochs=30,
    batch=16,
    imgsz=640,
    lr0=0.001,            # Lower LR for fine-tuning
    lrf=0.01,
    patience=10,
    project="/content/training",
    name="finetune_v1",
    hsv_v=0.3,
    degrees=3.0,
    mosaic=0.5,
    mixup=0.0,
)

print("\nFine-tuning complete!")

## Step 6: Test on Sample Images

In [ ]:
import glob
import random
from IPython.display import Image, display

# Pick best model (fine-tuned if exists, otherwise base)
ft_path = "/content/training/finetune_v1/weights/best.pt"
base_path = "/content/training/base_v1/weights/best.pt"
model_path = ft_path if os.path.exists(ft_path) else base_path

test_model = YOLO(model_path)
print(f"Using model: {model_path}")

# Get random test images
test_images = glob.glob("/content/merged_dataset/test/images/*")
samples = random.sample(test_images, min(8, len(test_images)))

# Run inference
results = test_model.predict(
    source=samples,
    conf=0.25,
    save=True,
    project="/content/predictions",
    name="test_samples",
)

# Display predictions
pred_images = sorted(glob.glob("/content/predictions/test_samples/*.jpg"))
for img_path in pred_images:
    display(Image(filename=img_path, width=600))
    print()

## Step 7: Download Trained Model

Download `trackify_behavior.pt` and put it in your project folder:
```
C:\Users\FARES\Downloads\trackify-eye-main\trackify_behavior.pt
```

In [ ]:
import shutil
from google.colab import files

# Pick best model
ft_path = "/content/training/finetune_v1/weights/best.pt"
base_path = "/content/training/base_v1/weights/best.pt"
src = ft_path if os.path.exists(ft_path) else base_path

# Copy with deployment name
dst = "/content/trackify_behavior.pt"
shutil.copy2(src, dst)

size_mb = os.path.getsize(dst) / (1024 * 1024)
print(f"Model ready: trackify_behavior.pt ({size_mb:.1f} MB)")
print(f"\nDownloading...")

# Auto-download to your computer
files.download(dst)

## Done!

After downloading `trackify_behavior.pt`:

1. Put it in `C:\Users\FARES\Downloads\trackify-eye-main\`
2. The backend code will automatically use it
3. Restart the project: `npm run dev:all`

The model detects: **person, phone_use, sleeping, cheating, fighting**

Classes NOT in the model (using fallbacks):
- **talking** → MediaPipe MAR detection
- **eating/drinking** → COCO bottle/cup detection